# Olist Customer Growth Intelligence — pipeline v2 (scoring-time aligned)

**Run on Kaggle:** *Add Input* → "Brazilian E-Commerce Public Dataset by Olist" → **Run All**.
Outputs in `/kaggle/working`: `retention_model.joblib`, `customers_scored.csv`, `customer_intelligence.csv`.

**What changed vs v1:** sections 1-4 are unchanged. Section 5 now builds the model the way the app uses it:
a customer is scored **once delivery is complete and the review has been answered**; only customers who have **not yet repeated at that moment** are "at risk";
the outcome is a repeat order in the **180 days after the scoring moment**; train/test splits are by scoring date.

**Definitions:** customer = `customer_unique_id`; value order = not canceled/unavailable; orders < 1 day after the first order are the same checkout (split cart), not a return visit.
Everything is **observational**: say "associated with", never "causes".

In [1]:
import os, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

SEED, WINDOW_DAYS, MIN_GAP_DAYS = 42, 180, 1.0
TEST_START = pd.Timestamp("2017-10-01")          # out-of-time test cohort starts here
TARGET = "repeat_180d"
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    cands = [Path(os.environ["OLIST_DIR"])] if os.environ.get("OLIST_DIR") else []
    for root, _, files in os.walk("/kaggle/input"):
        if "olist_orders_dataset.csv" in files:
            cands.append(Path(root)); break
    cands.append(Path("."))
    for c in cands:
        if (c / "olist_orders_dataset.csv").exists():
            return c
    raise FileNotFoundError("Attach the Olist dataset as a notebook input (or set OLIST_DIR).")

DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR, "| Output directory:", OUT_DIR)

def wilson(k, n):
    lo, hi = proportion_confint(k, n, method="wilson") if n > 0 else (np.nan, np.nan)
    return lo * 100, hi * 100

def rate_table(df, by, target=TARGET):
    t = df.groupby(by, observed=True)[target].agg(customers="size", repeaters="sum").reset_index()
    t["rate_pct"] = (t.repeaters / t.customers * 100).round(2)
    ci = [wilson(k, n) for k, n in zip(t.repeaters, t.customers)]
    t["ci_low"] = [round(c[0], 2) for c in ci]; t["ci_high"] = [round(c[1], 2) for c in ci]
    return t.rename(columns={by: "band"})

def attach(left, right, on="order_id"):
    """Left-join that refuses overlapping columns and row multiplication (grain guard)."""
    dup = [c for c in right.columns if c in left.columns and c != on]
    assert not dup, f"overlapping columns {dup}"
    out = left.merge(right, on=on, how="left", validate="many_to_one")
    assert len(out) == len(left), "join changed row count"
    return out

Data directory: /kaggle/input/datasets/olistbr/brazilian-ecommerce | Output directory: /kaggle/working


In [2]:
rd = lambda name: pd.read_csv(DATA_DIR / name)
customers, orders = rd("olist_customers_dataset.csv"), rd("olist_orders_dataset.csv")
items, pay, reviews = rd("olist_order_items_dataset.csv"), rd("olist_order_payments_dataset.csv"), rd("olist_order_reviews_dataset.csv")
products = rd("olist_products_dataset.csv")
try:
    trans = rd("product_category_name_translation.csv")
except FileNotFoundError:
    trans = None

for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c])
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])

orders = orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]],
                      on="customer_id", how="left", validate="many_to_one")
assert orders.customer_unique_id.notna().all() and orders.order_id.is_unique
orders["is_value_order"] = ~orders.order_status.isin(["canceled", "unavailable"])

vo = orders[orders.is_value_order].sort_values(["customer_unique_id", "order_purchase_timestamp", "order_id"]).reset_index(drop=True)
assert vo.order_id.is_unique
print(f"All orders: {len(orders):,} | value orders: {len(vo):,} | value customers: {vo.customer_unique_id.nunique():,} "
      f"| all customers: {customers.customer_unique_id.nunique():,}")

All orders: 99,441 | value orders: 98,207 | value customers: 94,990 | all customers: 96,096


## 1. Purchase occasions, repeat type and the observation window

In [3]:
g = vo.groupby("customer_unique_id")["order_purchase_timestamp"]
vo["order_seq"] = g.cumcount() + 1
vo["days_since_first"] = (vo.order_purchase_timestamp - g.transform("min")).dt.total_seconds() / 86400
vo["interpurchase_days"] = g.diff().dt.total_seconds() / 86400

first = vo[vo.order_seq == 1].copy()
assert first.customer_unique_id.is_unique

second = vo.loc[vo.order_seq == 2, ["customer_unique_id", "days_since_first"]].rename(columns={"days_since_first": "days_to_second_order"})
def next_after(t, name):
    return (vo[vo.days_since_first >= t].groupby("customer_unique_id")["days_since_first"].min().rename(name).reset_index())

first = (first.merge(second, on="customer_unique_id", how="left", validate="one_to_one")
              .merge(next_after(MIN_GAP_DAYS, "days_to_next_occasion"), on="customer_unique_id", how="left", validate="one_to_one")
              .merge(next_after(7.0, "days_to_next_7d"), on="customer_unique_id", how="left", validate="one_to_one"))

first["repeat_type"] = np.select(
    [first.days_to_next_occasion.notna(), first.days_to_second_order.notna()],
    ["True multi-occasion repeat", "Split-cart / same-day only"], default="Single purchase")
print(first.repeat_type.value_counts().to_frame("customers").assign(share_pct=lambda d: (d.customers / len(first) * 100).round(2)))

hidden = ((first.days_to_second_order < MIN_GAP_DAYS) & first.days_to_next_occasion.notna()).sum()
print(f"\nCustomers whose 2nd order was same-day but who DID return later: {hidden:,} "
      "(a 'second-order gap >= 1 day' rule would wrongly call these non-retained; the occasion rule counts them).")

                            customers  share_pct
repeat_type                                     
Single purchase                 92102      96.96
True multi-occasion repeat       2040       2.15
Split-cart / same-day only        848       0.89

Customers whose 2nd order was same-day but who DID return later: 48 (a 'second-order gap >= 1 day' rule would wrongly call these non-retained; the occasion rule counts them).


In [4]:
OBS_END_VALUE = vo.order_purchase_timestamp.max()          # primary: last value order
OBS_END_ALL = orders.order_purchase_timestamp.max()        # alternative: includes canceled/unavailable orders
print("Observation end (value orders):", OBS_END_VALUE, "| (all orders):", OBS_END_ALL)

def with_outcomes(f, obs_end):
    d = f.copy()
    d["eligible_180d"] = d.order_purchase_timestamp <= obs_end - pd.Timedelta(days=WINDOW_DAYS)
    d["repeat_raw"] = d.days_to_second_order.le(WINDOW_DAYS)
    d["repeat_2nd_rule"] = d.days_to_second_order.between(MIN_GAP_DAYS, WINDOW_DAYS)
    d[TARGET] = d.days_to_next_occasion.le(WINDOW_DAYS)
    d["repeat_7d"] = d.days_to_next_7d.le(WINDOW_DAYS)
    return d

rows = []
for obs_label, obs_end in [("value-order max (PRIMARY)", OBS_END_VALUE), ("all-order max", OBS_END_ALL)]:
    e = with_outcomes(first, obs_end); e = e[e.eligible_180d]
    for name, col in [("raw: any 2nd order <=180d", "repeat_raw"), ("2nd-order gap >=1d", "repeat_2nd_rule"),
                      ("occasion rule >=1d (PRIMARY)", TARGET), ("occasion rule >=7d", "repeat_7d")]:
        k, n = int(e[col].sum()), len(e); lo, hi = wilson(k, n)
        rows.append(dict(observation_end=obs_label, definition=name, eligible=n, repeaters=k,
                         rate_pct=round(k / n * 100, 3), ci_low=round(lo, 2), ci_high=round(hi, 2)))
sensitivity = pd.DataFrame(rows)
sensitivity

Observation end (value orders): 2018-09-03 09:06:57 | (all orders): 2018-10-17 17:30:18


,observation_end,definition,eligible,repeaters,rate_pct,ci_low,ci_high
0,value-order max (PRIMARY),raw: any 2nd order <=180d,58101,1832,3.153,3.01,3.30
1,value-order max (PRIMARY),2nd-order gap >=1d,58101,1092,1.879,1.77,1.99
2,value-order max (PRIMARY),occasion rule >=1d (PRIMARY),58101,1132,1.948,1.84,2.06
3,value-order max (PRIMARY),occasion rule >=7d,58101,1034,1.780,1.68,1.89
4,all-order max,raw: any 2nd order <=180d,68017,2042,3.002,2.88,3.13
5,all-order max,2nd-order gap >=1d,68017,1242,1.826,1.73,1.93
6,all-order max,occasion rule >=1d (PRIMARY),68017,1282,1.885,1.79,1.99
7,all-order max,occasion rule >=7d,68017,1167,1.716,1.62,1.82


## 2. First-order features (one row per customer)

In [5]:
it = items.sort_values(["order_id", "order_item_id"])
order_items = it.groupby("order_id").agg(first_order_items=("order_item_id", "count"),
                                         first_order_unique_products=("product_id", "nunique"),
                                         items_value=("price", "sum"), freight_value=("freight_value", "sum")).reset_index()
order_spend = pay.groupby("order_id")["payment_value"].sum().rename("first_order_spend").reset_index()
order_ptype = pay.sort_values(["order_id", "payment_value"], ascending=[True, False]).drop_duplicates("order_id")[["order_id", "payment_type"]]
order_review = reviews.groupby("order_id").agg(first_order_csat=("review_score", "mean"), review_created=("review_creation_date", "min")).reset_index()

top = it.drop_duplicates("order_id")[["order_id", "product_id"]].merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
if trans is not None:
    top = top.merge(trans, on="product_category_name", how="left"); cat_col = "product_category_name_english"
else:
    cat_col = "product_category_name"
CONSUMABLE = {"health_beauty", "perfumery", "pet_shop", "food", "food_drink", "drinks"}   # heuristic, see notes
top["category_type"] = np.where(top[cat_col].isna(), "Unknown", np.where(top[cat_col].isin(CONSUMABLE), "Consumable", "Durable/other"))
order_cat = top[["order_id", "category_type"]]

fo = first.copy()
for right in (order_items, order_spend, order_ptype, order_review, order_cat):
    fo = attach(fo, right)

den = fo.items_value + fo.freight_value
fo["multi_product"] = (fo.first_order_unique_products >= 2).astype(float).where(fo.first_order_unique_products.notna())
fo["log_spend"] = np.log1p(fo.first_order_spend)
fo["freight_share"] = (fo.freight_value / den).where(den > 0)
fo["transit_days"] = (fo.order_delivered_customer_date - fo.order_purchase_timestamp).dt.total_seconds() / 86400
delay = (fo.order_delivered_customer_date - fo.order_estimated_delivery_date).dt.total_seconds() / 86400
fo["is_delivery_late"] = (delay > 0).astype(float).where(delay.notna())
fo["is_sp"] = (fo.customer_state == "SP").astype(float)
fo["acq_quarter"] = fo.order_purchase_timestamp.dt.to_period("Q").astype(str)

CLIPS = {"first_order_items": (1, 10), "transit_days": (0, 60), "freight_share": (0, 1)}   # same clips are applied at scoring time
for c, (lo, hi) in CLIPS.items():
    fo[c] = fo[c].clip(lo, hi)

fo = with_outcomes(fo, OBS_END_VALUE)
for c in (TARGET, "repeat_raw", "repeat_2nd_rule", "repeat_7d"):
    fo[c] = fo[c].astype(int)
assert fo.customer_unique_id.is_unique and len(fo) == len(first)

cohort = fo[fo.eligible_180d].copy()
print(f"Eligible cohort: {len(cohort):,} customers | repeaters: {cohort[TARGET].sum():,} | rate: {cohort[TARGET].mean()*100:.2f}%")

rep = cohort[cohort[TARGET] == 1]
repeat_ts = rep.order_purchase_timestamp + pd.to_timedelta(rep.days_to_next_occasion, unit="D")
has_rev = rep.review_created.notna()
print(f"Timing check: {(repeat_ts[has_rev] < rep.review_created[has_rev]).mean()*100:.1f}% of repeaters (with a review) ordered again BEFORE "
      "the first-order review was written - CSAT/delivery features are only known after delivery, so scores apply post-delivery.")

Eligible cohort: 58,101 customers | repeaters: 1,132 | rate: 1.95%
Timing check: 14.7% of repeaters (with a review) ordered again BEFORE the first-order review was written - CSAT/delivery features are only known after delivery, so scores apply post-delivery.


## 3. Which first-order attributes are associated with 180-day repeat? (Wilson 95% CIs)

In [6]:
c = cohort.copy()
c["Basket volume"] = pd.cut(c.first_order_items, [0, 1, 2, np.inf], labels=["1 item", "2 items", "3+ items"])
c["Product diversity"] = pd.cut(c.first_order_unique_products, [0, 1, 2, np.inf], labels=["1 product", "2 products", "3+ products"])
c["First-order spend"] = pd.qcut(c.first_order_spend, 3, labels=["Low", "Medium", "High"], duplicates="drop")
c["Review score"] = pd.cut(c.first_order_csat, [0, 2, 3, 4, 5], labels=["1-2", "3", "4", "5"])
c["Transit time"] = pd.cut(c.transit_days, [-np.inf, 7, 14, 21, np.inf], labels=["<7d", "7-14d", "14-21d", ">21d"])
c["Delivered late vs estimate"] = c.is_delivery_late.map({0.0: "On time", 1.0: "Late"})
c["Product type (heuristic)"] = c.category_type
c["Payment type"] = c.payment_type
c["Customer state"] = np.where(c.is_sp == 1, "SP", "Other")

dims = ["Basket volume", "Product diversity", "First-order spend", "Review score", "Transit time",
        "Delivered late vs estimate", "Product type (heuristic)", "Payment type", "Customer state"]
triad = pd.concat([rate_table(c, d).assign(dimension=d) for d in dims], ignore_index=True)
triad[["dimension", "band", "customers", "repeaters", "rate_pct", "ci_low", "ci_high"]]

,dimension,band,customers,repeaters,rate_pct,ci_low,ci_high
0,Basket volume,1 item,52376,981,1.87,1.76,1.99
1,Basket volume,2 items,4419,104,2.35,1.95,2.84
2,Basket volume,3+ items,1298,47,3.62,2.73,4.78
3,Product diversity,1 product,56253,1077,1.91,1.80,2.03
4,Product diversity,2 products,1634,49,3.00,2.28,3.94
5,Product diversity,3+ products,206,6,2.91,1.34,6.21
6,First-order spend,Low,19398,385,1.98,1.80,2.19
7,First-order spend,Medium,19338,360,1.86,1.68,2.06
8,First-order spend,High,19364,387,2.00,1.81,2.21
9,Review score,1-2,8469,136,1.61,1.36,1.90


In [7]:
cols = [TARGET, "first_order_items", "multi_product", "log_spend", "freight_share", "first_order_csat",
        "transit_days", "is_delivery_late", "is_sp", "acq_quarter"]
md_ = cohort.dropna(subset=cols).copy()
md_["transit_weeks"] = md_.transit_days / 7
fml = (f"{TARGET} ~ first_order_items + multi_product + log_spend + freight_share + first_order_csat + "
       "transit_weeks + is_delivery_late + is_sp + C(acq_quarter)")
res = smf.logit(fml, md_).fit(disp=False, maxiter=200)
ci = np.exp(res.conf_int())
odds = pd.DataFrame({"odds_ratio": np.exp(res.params), "ci_low": ci[0], "ci_high": ci[1], "p_value": res.pvalues}).round(4)
print(f"Rows in model: {len(md_):,} | events: {int(md_[TARGET].sum()):,} | acquisition-quarter dummies omitted from the table")
odds[~odds.index.str.startswith(("C(", "Intercept"))]

Rows in model: 56,507 | events: 1,108 | acquisition-quarter dummies omitted from the table


,odds_ratio,ci_low,ci_high,p_value
first_order_items,1.2250,1.1023,1.3612,0.0002
multi_product,1.2864,0.9459,1.7494,0.1084
log_spend,0.9743,0.8604,1.1033,0.6817
freight_share,1.2410,0.5761,2.6736,0.5813
first_order_csat,1.0677,1.0125,1.1259,0.0156
transit_weeks,0.9338,0.8669,1.0060,0.0713
is_delivery_late,1.2679,0.9259,1.7362,0.1390
is_sp,1.1423,0.9956,1.3106,0.0579


## 4. Descriptive behavioural segments (restored; same method as before, order-independent labels)

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ids = customers[["customer_unique_id"]].drop_duplicates()
cust = orders.groupby("customer_unique_id").agg(total_orders=("order_id", "nunique"), value_orders=("is_value_order", "sum")).reset_index()
vmap = vo[["order_id", "customer_unique_id"]]
ci_ = items.merge(vmap, on="order_id").groupby("customer_unique_id").agg(total_items=("order_item_id", "count"), unique_products_purchased=("product_id", "nunique")).reset_index()
cp_ = pay.merge(vmap, on="order_id").groupby("customer_unique_id").agg(total_payment_value=("payment_value", lambda s: s.sum(min_count=1))).reset_index()
ip_ = vo.groupby("customer_unique_id").agg(median_interpurchase_days=("interpurchase_days", "median")).reset_index()

prof = ids.merge(cust, on="customer_unique_id", how="left", validate="one_to_one")
for r in (ci_, cp_, ip_):
    prof = prof.merge(r, on="customer_unique_id", how="left", validate="one_to_one")
nv = prof.value_orders == 0
prof.loc[nv, ["total_items", "unique_products_purchased", "total_payment_value"]] = 0
prof["log_total_payment_value"] = np.log1p(prof.total_payment_value)
assert prof.customer_unique_id.is_unique
print(f"Customers: {len(prof):,} | no-value-order: {int(nv.sum()):,}")

A = ["value_orders", "total_items", "unique_products_purchased", "log_total_payment_value"]
sa = prof[prof[A].notna().all(axis=1)].copy()
Xa = StandardScaler().fit_transform(sa[A])
for k in (2, 3, 4):
    lab = KMeans(k, random_state=SEED, n_init=20).fit_predict(Xa)
    print(f"Stage A k={k} silhouette (10k sample): {silhouette_score(Xa, lab, sample_size=10000, random_state=SEED):.4f}")
sa["cluster"] = KMeans(3, random_state=SEED, n_init=20).fit_predict(Xa)

ga = sa.groupby("cluster").agg(zero_share=("value_orders", lambda s: (s == 0).mean()), pay=("total_payment_value", "mean"))
c_novalue = ga.zero_share.idxmax()
rest = ga.drop(c_novalue).pay.sort_values()
c_small, c_large = rest.index[0], rest.index[-1]

rep_df = prof[(prof.value_orders >= 2)].copy()
rep_df["log_median_interpurchase_days"] = np.log1p(rep_df.median_interpurchase_days)
B = ["value_orders", "log_median_interpurchase_days", "total_items", "unique_products_purchased", "log_total_payment_value"]
sb = rep_df[rep_df[B].notna().all(axis=1)].copy()
Xb = StandardScaler().fit_transform(sb[B])
sb["cluster_b"] = KMeans(2, random_state=SEED, n_init=20).fit_predict(Xb)
c_high = sb.groupby("cluster_b").total_payment_value.mean().idxmax()

seg = prof[["customer_unique_id", "value_orders"]].merge(sa[["customer_unique_id", "cluster"]], on="customer_unique_id", how="left") \
                                                   .merge(sb[["customer_unique_id", "cluster_b"]], on="customer_unique_id", how="left")
seg["final_segment"] = "Data Incomplete"
seg.loc[seg.value_orders == 0, "final_segment"] = "No Value Order"
one = seg.value_orders == 1
seg.loc[one & (seg.cluster == c_small), "final_segment"] = "Single Purchase - Small Basket"
seg.loc[one & (seg.cluster == c_large), "final_segment"] = "Single Purchase - Larger Basket"
seg.loc[(seg.value_orders >= 2) & (seg.cluster_b == c_high), "final_segment"] = "Repeat Purchase - Higher Intensity"
seg.loc[(seg.value_orders >= 2) & seg.cluster_b.notna() & (seg.cluster_b != c_high), "final_segment"] = "Repeat Purchase - Standard"

reference = {"No Value Order": 1106, "Single Purchase - Small Basket": 88607, "Single Purchase - Larger Basket": 3487,
             "Repeat Purchase - Standard": 2576, "Repeat Purchase - Higher Intensity": 312, "Data Incomplete": 8}
dist = seg.final_segment.value_counts().to_frame("customers")
dist["share_pct"] = (dist.customers / len(seg) * 100).round(3)
dist["earlier_run"] = pd.Series(reference)
print(f"Total {len(seg):,} (earlier run: 96,096). Differences vs the earlier run would indicate library-version effects.")
dist

Customers: 96,096 | no-value-order: 1,106
Stage A k=2 silhouette (10k sample): 0.7385
Stage A k=3 silhouette (10k sample): 0.7570
Stage A k=4 silhouette (10k sample): 0.4727
Total 96,096 (earlier run: 96,096). Differences vs the earlier run would indicate library-version effects.


,customers,share_pct,earlier_run
final_segment,,,
Single Purchase - Small Basket,88607,92.207,88607
Single Purchase - Larger Basket,3487,3.629,3487
Repeat Purchase - Standard,2576,2.681,2576
No Value Order,1106,1.151,1106
Repeat Purchase - Higher Intensity,312,0.325,312
Data Incomplete,8,0.008,8


In [9]:
intel = prof.merge(seg[["customer_unique_id", "final_segment"]], on="customer_unique_id", how="left", validate="one_to_one") \
            .merge(first[["customer_unique_id", "repeat_type", "days_to_second_order", "days_to_next_occasion"]], on="customer_unique_id", how="left", validate="one_to_one")
intel["repeat_type"] = intel.repeat_type.fillna("No value order")
assert intel.customer_unique_id.is_unique and intel.final_segment.notna().all()
intel.to_csv(OUT_DIR / "customer_intelligence.csv", index=False)
print("Segment x repeat type (are 'repeat' segments really repeat, or split-cart only?)")
pd.crosstab(intel.final_segment, intel.repeat_type, margins=True)

Segment x repeat type (are 'repeat' segments really repeat, or split-cart only?)


repeat_type,No value order,Single purchase,Split-cart / same-day only,True multi-occasion repeat,All
final_segment,,,,,
Data Incomplete,0,8,0,0,8
No Value Order,1106,0,0,0,1106
Repeat Purchase - Higher Intensity,0,0,53,259,312
Repeat Purchase - Standard,0,0,795,1781,2576
Single Purchase - Larger Basket,0,3487,0,0,3487
Single Purchase - Small Basket,0,88607,0,0,88607
All,1106,92102,848,2040,96096


## 5. v2 population: who is "at risk" at the moment of scoring?
Scoring time `T = max(delivery date, review answer time)`. Customers who already placed a returning order (≥ 1 day after the first) **before T** are removed:
they are already retained, and their review could not have influenced that purchase. Outcome = a returning order in (T, T + 180 days].

In [10]:
TARGET_V2 = "repeat_after_score_180d"
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])
rv_ans = reviews.groupby("order_id").agg(review_answered=("review_answer_timestamp", "max"),
                                          has_review_comment=("review_comment_message", lambda s: bool(s.notna().any()))).reset_index()
fo2 = attach(fo, rv_ans)

both = fo2.order_delivered_customer_date.notna() & fo2.review_answered.notna()
fo2["scoring_ts"] = fo2[["order_delivered_customer_date", "review_answered"]].max(axis=1).where(both)
fo2["days_to_score"] = (fo2.scoring_ts - fo2.order_purchase_timestamp).dt.total_seconds() / 86400

occ = vo.loc[vo.days_since_first >= MIN_GAP_DAYS, ["customer_unique_id", "order_purchase_timestamp"]].rename(columns={"order_purchase_timestamp": "occ_ts"})
occ = occ.merge(fo2[["customer_unique_id", "scoring_ts"]], on="customer_unique_id", how="left", validate="many_to_one")
occ["delta_days"] = (occ.occ_ts - occ.scoring_ts).dt.total_seconds() / 86400
before = occ[occ.delta_days <= 0].groupby("customer_unique_id").size().rename("n_returns_before_score").reset_index()
after = occ[occ.delta_days > 0].groupby("customer_unique_id")["delta_days"].min().rename("days_to_repeat_after_score").reset_index()
fo2 = fo2.merge(before, on="customer_unique_id", how="left", validate="one_to_one").merge(after, on="customer_unique_id", how="left", validate="one_to_one")
fo2["already_repeated"] = fo2.n_returns_before_score.fillna(0) > 0
fo2["at_risk"] = fo2.scoring_ts.notna() & ~fo2.already_repeated
fo2["eligible_v2"] = fo2.at_risk & (fo2.scoring_ts <= OBS_END_VALUE - pd.Timedelta(days=WINDOW_DAYS))
fo2[TARGET_V2] = fo2.days_to_repeat_after_score.le(WINDOW_DAYS).astype(int)
fo2["is_consumable"] = (fo2.category_type == "Consumable").astype(float)
fo2["paid_voucher"] = (fo2.payment_type == "voucher").astype(float)
fo2["has_review_comment"] = fo2.has_review_comment.astype(float)
assert fo2.customer_unique_id.is_unique and len(fo2) == len(fo)

funnel = pd.DataFrame([
    ("First orders (value customers)", len(fo2), None),
    ("... with scoring time known (delivered + review answered)", int(fo2.scoring_ts.notna().sum()), None),
    ("... already returned BEFORE scoring (excluded: not at risk)", int(fo2.already_repeated.sum()), None),
    ("... at risk at scoring time", int(fo2.at_risk.sum()), None),
    ("... with >=180 days of follow-up after scoring (ELIGIBLE v2)", int(fo2.eligible_v2.sum()), fo2.loc[fo2.eligible_v2, TARGET_V2].mean() * 100),
    ("v1 eligible cohort (from first order) for comparison", len(cohort), cohort[TARGET].mean() * 100)],
    columns=["step", "customers", "repeat_rate_pct"])
funnel["repeat_rate_pct"] = funnel.repeat_rate_pct.round(2)
funnel

,step,customers,repeat_rate_pct
0,First orders (value customers),94990,NaN
1,... with scoring time known (delivered + revie...,92678,NaN
2,... already returned BEFORE scoring (excluded:...,343,NaN
3,... at risk at scoring time,92335,NaN
4,... with >=180 days of follow-up after scoring...,51876,1.74
5,v1 eligible cohort (from first order) for comp...,58101,1.95


### 5a. Are the associations robust under the v2 definition? (adjusted odds ratios, v1 vs v2)

In [11]:
cols2 = [TARGET_V2, "first_order_items", "multi_product", "log_spend", "freight_share", "first_order_csat",
         "transit_days", "is_delivery_late", "is_sp", "acq_quarter"]
m2 = fo2[fo2.eligible_v2].dropna(subset=cols2).copy(); m2["transit_weeks"] = m2.transit_days / 7
res2 = smf.logit(fml.replace(TARGET, TARGET_V2), m2).fit(disp=False, maxiter=200)
ci2 = np.exp(res2.conf_int())
odds2 = pd.DataFrame({"OR_v2": np.exp(res2.params), "ci_low_v2": ci2[0], "ci_high_v2": ci2[1], "p_v2": res2.pvalues}).round(4)
cmp_ = odds.rename(columns={"odds_ratio": "OR_v1", "ci_low": "ci_low_v1", "ci_high": "ci_high_v1", "p_value": "p_v1"}).join(odds2)
print(f"v2 model rows: {len(m2):,} | events: {int(m2[TARGET_V2].sum()):,}")
cmp_[~cmp_.index.str.startswith(("C(", "Intercept"))]

v2 model rows: 51,875 | events: 902


,OR_v1,ci_low_v1,ci_high_v1,p_v1,OR_v2,ci_low_v2,ci_high_v2,p_v2
first_order_items,1.2250,1.1023,1.3612,0.0002,1.1555,1.0136,1.3174,0.0307
multi_product,1.2864,0.9459,1.7494,0.1084,1.1440,0.7926,1.6513,0.4724
log_spend,0.9743,0.8604,1.1033,0.6817,0.9753,0.8494,1.1198,0.7230
freight_share,1.2410,0.5761,2.6736,0.5813,1.2586,0.5338,2.9679,0.5992
first_order_csat,1.0677,1.0125,1.1259,0.0156,1.0706,1.0080,1.1371,0.0265
transit_weeks,0.9338,0.8669,1.0060,0.0713,0.9123,0.8364,0.9952,0.0385
is_delivery_late,1.2679,0.9259,1.7362,0.1390,1.2936,0.8806,1.9003,0.1896
is_sp,1.1423,0.9956,1.3106,0.0579,1.1142,0.9571,1.2972,0.1631


## 6. v2 predictive model
Same 8 deployable features (so the app keeps working), but the population, outcome and splits follow the scoring moment.
Splits use the **scoring date**: test = scored on/after 2017-10-01; training B = customers whose full 180-day outcome was known before the test period began.

In [12]:
import joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

FEATURES = ["first_order_items", "multi_product", "log_spend", "freight_share", "first_order_csat", "transit_days", "is_delivery_late", "is_sp"]
EXTRA = ["is_consumable", "paid_voucher", "has_review_comment", "days_to_score"]
mdl = fo2[fo2.eligible_v2].dropna(subset=FEATURES + [TARGET_V2]).copy()
print(f"Model frame: {len(mdl):,} of {int(fo2.eligible_v2.sum()):,} eligible | events {int(mdl[TARGET_V2].sum()):,} | base rate {mdl[TARGET_V2].mean()*100:.2f}%")
GAP_END = TEST_START - pd.Timedelta(days=WINDOW_DAYS)

def make_pipe():
    return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])

def boot_auc(y, p, n=200):
    r = np.random.default_rng(SEED); idx = np.arange(len(y)); out = []
    for _ in range(n):
        s = r.choice(idx, len(idx))
        if y[s].min() != y[s].max():
            out.append(roc_auc_score(y[s], p[s]))
    return np.percentile(out, [2.5, 97.5])

def deciles(y, p):
    d = pd.DataFrame({"y": y, "p": p}); d["decile"] = 10 - pd.qcut(d.p.rank(method="first"), 10, labels=False)
    t = d.groupby("decile").agg(customers=("y", "size"), repeaters=("y", "sum"), mean_pred_pct=("p", lambda s: s.mean() * 100)).reset_index().sort_values("decile")
    t["actual_pct"] = (t.repeaters / t.customers * 100).round(2); t["mean_pred_pct"] = t.mean_pred_pct.round(2)
    t["lift"] = (t.actual_pct / (y.mean() * 100)).round(2)
    return t

def run_oot(train_mask, label, feats=FEATURES, factory=make_pipe, verbose=True):
    tr, te = mdl[train_mask], mdl[mdl.scoring_ts >= TEST_START]
    pipe = factory().fit(tr[feats], tr[TARGET_V2]); p = pipe.predict_proba(te[feats])[:, 1]; y = te[TARGET_V2].values
    lo, hi = boot_auc(y, p)
    r = dict(label=label, n_train=len(tr), events_train=int(tr[TARGET_V2].sum()), n_test=len(te), base_rate_test_pct=round(y.mean() * 100, 2),
             roc_auc=float(roc_auc_score(y, p)), roc_auc_ci=(float(lo), float(hi)), pr_auc=float(average_precision_score(y, p)),
             baselines={k: float(roc_auc_score(y, s)) for k, s in {"csat_only": te.first_order_csat, "items_only": te.first_order_items, "fast_transit_only": -te.transit_days}.items()},
             deciles=deciles(y, p), pipe=pipe, test=te.assign(p=p))
    if verbose:
        print(f"\n=== {label} ===\nTrain {r['n_train']:,} ({r['events_train']} events) | Test {r['n_test']:,} (base {r['base_rate_test_pct']}%)")
        print(f"ROC-AUC {r['roc_auc']:.3f} (95% bootstrap CI {lo:.3f}-{hi:.3f}) | PR-AUC {r['pr_auc']:.4f} (random = {y.mean():.4f})")
        print("Single-feature AUCs:", {k: round(v, 3) for k, v in r["baselines"].items()})
        display(r["deciles"])
    return r

oot_a = run_oot(mdl.scoring_ts < TEST_START, "OUT-OF-TIME A (train outcomes overlap test period)")
oot_b = run_oot(mdl.scoring_ts <= GAP_END, "OUT-OF-TIME B (gapped: train outcomes fully known before test)")

cv_p = cross_val_predict(make_pipe(), mdl[FEATURES], mdl[TARGET_V2], cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
cv_auc = float(roc_auc_score(mdl[TARGET_V2], cv_p)); cv_ci = boot_auc(mdl[TARGET_V2].values, cv_p)
print(f"\n5-fold CV ROC-AUC: {cv_auc:.3f} (95% bootstrap CI {cv_ci[0]:.3f}-{cv_ci[1]:.3f})")

Model frame: 51,875 of 51,876 eligible | events 902 | base rate 1.74%

=== OUT-OF-TIME A (train outcomes overlap test period) ===
Train 23,381 (459 events) | Test 28,494 (base 1.55%)
ROC-AUC 0.542 (95% bootstrap CI 0.516-0.571) | PR-AUC 0.0187 (random = 0.0155)
Single-feature AUCs: {'csat_only': 0.523, 'items_only': 0.514, 'fast_transit_only': 0.537}


,decile,customers,repeaters,mean_pred_pct,actual_pct,lift
0,1,2850,66,2.44,2.32,1.49
1,2,2849,44,2.16,1.54,0.99
2,3,2849,40,2.08,1.40,0.90
3,4,2850,43,2.01,1.51,0.97
4,5,2849,45,1.95,1.58,1.02
5,6,2849,52,1.88,1.83,1.18
6,7,2850,45,1.82,1.58,1.02
7,8,2849,44,1.73,1.54,0.99
8,9,2849,35,1.62,1.23,0.79
9,10,2850,29,1.35,1.02,0.66



=== OUT-OF-TIME B (gapped: train outcomes fully known before test) ===
Train 4,037 (65 events) | Test 28,494 (base 1.55%)
ROC-AUC 0.516 (95% bootstrap CI 0.490-0.544) | PR-AUC 0.0168 (random = 0.0155)
Single-feature AUCs: {'csat_only': 0.523, 'items_only': 0.514, 'fast_transit_only': 0.537}


,decile,customers,repeaters,mean_pred_pct,actual_pct,lift
0,1,2850,52,2.91,1.82,1.17
1,2,2849,48,2.10,1.68,1.08
2,3,2849,46,1.87,1.61,1.04
3,4,2850,40,1.71,1.40,0.90
4,5,2849,48,1.55,1.68,1.08
5,6,2849,38,1.41,1.33,0.86
6,7,2850,44,1.28,1.54,0.99
7,8,2849,41,1.15,1.44,0.93
8,9,2849,37,0.99,1.30,0.84
9,10,2850,49,0.71,1.72,1.11



5-fold CV ROC-AUC: 0.544 (95% bootstrap CI 0.527-0.560)


### 6a. Does anything beat the 8-feature logistic model? (evidence before changing it)

In [13]:
hgb = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, l2_regularization=1.0, random_state=SEED)
variants = [("Logistic, 8 features (deployed)", make_pipe, FEATURES),
            ("Logistic, + 4 extra features", make_pipe, FEATURES + EXTRA),
            ("Gradient boosting, 8 features", hgb, FEATURES),
            ("Gradient boosting, + 4 extra features", hgb, FEATURES + EXTRA)]
rows = []
for name, fac, feats in variants:
    d = mdl.dropna(subset=feats)
    p = cross_val_predict(fac(), d[feats], d[TARGET_V2], cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    lo, hi = boot_auc(d[TARGET_V2].values, p)
    rows.append(dict(model=name, n=len(d), cv_auc=round(roc_auc_score(d[TARGET_V2], p), 3), cv_ci=f"{lo:.3f}-{hi:.3f}", cv_ci_low=lo))
bench = pd.DataFrame(rows)
base_auc = bench.loc[0, "cv_auc"]
bench["beats_deployed"] = bench.cv_ci_low > base_auc
print("Rule: only change the deployed model if a variant's CV lower bound exceeds the deployed model's CV point estimate.")
bench.drop(columns="cv_ci_low")

Rule: only change the deployed model if a variant's CV lower bound exceeds the deployed model's CV point estimate.


,model,n,cv_auc,cv_ci,beats_deployed
0,"Logistic, 8 features (deployed)",51875,0.544,0.527-0.560,False
1,"Logistic, + 4 extra features",51875,0.551,0.535-0.569,False
2,"Gradient boosting, 8 features",51875,0.517,0.497-0.534,False
3,"Gradient boosting, + 4 extra features",51875,0.529,0.507-0.546,False


## 7. Export the v2 artifact and scored customers

In [14]:
final_pipe = make_pipe().fit(mdl[FEATURES], mdl[TARGET_V2])
p_all = final_pipe.predict_proba(mdl[FEATURES])[:, 1]
q = {f"p{int(x*100)}": float(np.quantile(p_all, x)) for x in (0.5, 0.8, 0.9)}
coefs = {k: float(v) for k, v in zip(FEATURES, final_pipe.named_steps["clf"].coef_[0].round(4))}

# Historical outcome per band, measured out-of-time with the gapped model (bands = top 10% / 50-90% / bottom 50% of its own training scores)
trn = mdl[mdl.scoring_ts <= GAP_END]
pt = oot_b["pipe"].predict_proba(trn[FEATURES])[:, 1]; qb = {"p50": np.quantile(pt, .5), "p90": np.quantile(pt, .9)}
tb = oot_b["test"].copy()
tb["band"] = np.where(tb.p >= qb["p90"], "High", np.where(tb.p >= qb["p50"], "Medium", "Low"))
band_outcomes = []
for b in ["High", "Medium", "Low"]:
    s = tb[tb.band == b]; k, n = int(s[TARGET_V2].sum()), len(s); lo, hi = wilson(k, n)
    band_outcomes.append({"band": b, "customers": n, "repeaters": k, "repeat_rate_pct": round(k / n * 100, 2), "ci_low": round(lo, 2), "ci_high": round(hi, 2)})
band_outcomes_df = pd.DataFrame(band_outcomes)
print("Out-of-time repeat rate by propensity band (test base rate %.2f%%):" % oot_b["base_rate_test_pct"])
display(band_outcomes_df)

strip = lambda r: {k: v for k, v in r.items() if k not in ("deciles", "pipe", "test")} | {"deciles": r["deciles"].to_dict("records")}
artifact = {
    "model_version": "olist-logit-v2-" + pd.Timestamp.now().strftime("%Y%m%d"),
    "pipeline": final_pipe, "features": FEATURES, "clips": CLIPS, "target": TARGET_V2,
    "window_days": WINDOW_DAYS, "min_gap_days": MIN_GAP_DAYS, "sklearn_version": sklearn.__version__,
    "n_train": int(len(mdl)), "n_events": int(mdl[TARGET_V2].sum()), "base_rate": float(mdl[TARGET_V2].mean()),
    "score_quantiles": q, "coefficients_standardized": coefs,
    "metrics": {"cv_auc": cv_auc, "cv_auc_ci": [float(x) for x in cv_ci], "oot_a": strip(oot_a), "oot_b": strip(oot_b),
                "band_outcomes_oot_b": band_outcomes, "benchmarks": bench.drop(columns="cv_ci_low").to_dict("records")},
    "population": "customers who have NOT yet returned at scoring time; outcome = returning order (>=1 day after first order) within 180 days after scoring",
    "scoring_time": "after delivery is complete and the first-order review has been answered",
    "caveats": ["Observational associations; no intervention effect is estimated.",
                "Probabilities are 180-day repeat propensities for a low base-rate event; use as relative bands.",
                "Trained on one marketplace and period; re-validate before reuse."],
}
joblib.dump(artifact, OUT_DIR / "retention_model.joblib")
chk = joblib.load(OUT_DIR / "retention_model.joblib")
assert np.allclose(chk["pipeline"].predict_proba(mdl[FEATURES].head(100))[:, 1], p_all[:100]), "reload mismatch"
print("Saved retention_model.joblib | reload OK | version", artifact["model_version"], "| scikit-learn", sklearn.__version__)

sc = fo2.dropna(subset=FEATURES).copy()
sc["p_repeat"] = final_pipe.predict_proba(sc[FEATURES])[:, 1]
sc["risk_band"] = np.where(sc.p_repeat >= q["p90"], "High", np.where(sc.p_repeat >= q["p50"], "Medium", "Low"))
sc[TARGET_V2] = sc[TARGET_V2].where(sc.eligible_v2)
sc = sc.merge(intel[["customer_unique_id", "final_segment"]], on="customer_unique_id", how="left", validate="one_to_one")
keep = (["customer_unique_id", "order_id", "order_purchase_timestamp", "scoring_ts", "customer_state", "final_segment", "repeat_type"] + FEATURES
        + ["category_type", "payment_type", "p_repeat", "risk_band", "at_risk", "already_repeated", "eligible_v2", TARGET_V2])
sc[keep].to_csv(OUT_DIR / "customers_scored.csv", index=False)
print(f"Saved customers_scored.csv ({len(sc):,} rows; {int(sc.at_risk.sum()):,} at risk at scoring time)")

Out-of-time repeat rate by propensity band (test base rate 1.55%):


,band,customers,repeaters,repeat_rate_pct,ci_low,ci_high
0,High,2507,41,1.64,1.21,2.21
1,Medium,10959,182,1.66,1.44,1.92
2,Low,15028,220,1.46,1.28,1.67


Saved retention_model.joblib | reload OK | version olist-logit-v2-20261002 | scikit-learn 1.6.1
Saved customers_scored.csv (92,677 rows; 92,334 at risk at scoring time)


## 8. Auto-generated summary (numbers come from this run; nothing hard-coded)

In [15]:
prim = sensitivity[(sensitivity.observation_end.str.contains("PRIMARY")) & (sensitivity.definition.str.contains("PRIMARY"))].iloc[0]
v2_rate = mdl[TARGET_V2].mean() * 100
sig = cmp_[~cmp_.index.str.startswith(("C(", "Intercept"))]
print(f"1) v1 definition (from first order): {prim.rate_pct:.2f}% repeat. v2 definition (from scoring time, not yet returned): {v2_rate:.2f}% on {len(mdl):,} at-risk customers.")
print(f"   {int(fo2.already_repeated.sum()):,} customers had already returned before their scoring moment and are no longer treated as at risk.")
print("2) Terms with p<0.05 under v2:", sig[sig.p_v2 < 0.05].index.tolist() or "none", "| under v1:", sig[sig.p_v1 < 0.05].index.tolist() or "none")
tb_ = oot_b["deciles"].iloc[0]
print(f"3) v2 discrimination: CV AUC {cv_auc:.3f} ({cv_ci[0]:.3f}-{cv_ci[1]:.3f}); gapped OOT AUC {oot_b['roc_auc']:.3f} ({oot_b['roc_auc_ci'][0]:.3f}-{oot_b['roc_auc_ci'][1]:.3f}); "
      f"top decile {tb_.actual_pct}% vs {oot_b['base_rate_test_pct']}% baseline ({tb_.lift}x).")
hi_ = band_outcomes_df.iloc[0]; lo_ = band_outcomes_df.iloc[2]
print(f"4) Out-of-time: High band repeated at {hi_.repeat_rate_pct}% ({hi_.ci_low}-{hi_.ci_high}), Low band at {lo_.repeat_rate_pct}% ({lo_.ci_low}-{lo_.ci_high}).")
better = bench[bench.beats_deployed].model.tolist()
print("5) Variants that beat the deployed model:", better or "none - keep the 8-feature logistic model.")
print("6) Not established: that any intervention changes repeat behaviour. Use the holdout arm to measure it.")

1) v1 definition (from first order): 1.95% repeat. v2 definition (from scoring time, not yet returned): 1.74% on 51,875 at-risk customers.
   343 customers had already returned before their scoring moment and are no longer treated as at risk.
2) Terms with p<0.05 under v2: ['first_order_items', 'first_order_csat', 'transit_weeks'] | under v1: ['first_order_items', 'first_order_csat']
3) v2 discrimination: CV AUC 0.544 (0.527-0.560); gapped OOT AUC 0.516 (0.490-0.544); top decile 1.82% vs 1.55% baseline (1.17x).
4) Out-of-time: High band repeated at 1.64% (1.21-2.21), Low band at 1.46% (1.28-1.67).
5) Variants that beat the deployed model: none - keep the 8-feature logistic model.
6) Not established: that any intervention changes repeat behaviour. Use the holdout arm to measure it.
